In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS automobile_repair.silver;

CREATE OR REPLACE TABLE automobile_repair.silver.invoice AS
WITH deduplicated_invoices AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY invoice_id ORDER BY invoice_date DESC) as rn
    FROM automobile_repair.bronze.invoice
),
base_invoices AS (
    SELECT
        invoice_id,
        order_id,
        CAST(invoice_date AS DATE) AS invoice_date,
        invoice_amount
    FROM deduplicated_invoices
    WHERE rn = 1
      AND invoice_amount IS NOT NULL  -- Exclude NULL amounts (though none exist)
)
SELECT 
    -- Identifiers (2)
    i.invoice_id,
    i.order_id,
    
    -- Revenue data (2)
    i.invoice_date,
    i.invoice_amount,
    
    -- Time dimensions (2)
    YEAR(i.invoice_date) AS invoice_year,
    MONTH(i.invoice_date) AS invoice_month,
    
    -- Store & Manager context (3)
    o.store_id,
    s.manager_id,
    s.manager_name
    
FROM base_invoices i
LEFT JOIN automobile_repair.bronze.order o ON i.order_id = o.order_id
LEFT JOIN automobile_repair.bronze.store s ON o.store_id = s.store_id
ORDER BY i.invoice_date DESC, i.invoice_id;

In [0]:
%sql
-- Monthly revenue by store and manager
-- Ready for MTD Performance and Revenue vs Budget KPIs
SELECT 
    invoice_year,
    invoice_month,
    store_id,
    manager_name,
    COUNT(DISTINCT order_id) as total_orders,
    COUNT(*) as total_invoices,
    SUM(invoice_amount) as total_revenue,
    ROUND(AVG(invoice_amount), 2) as avg_invoice_amount
FROM automobile_repair.silver.invoice
GROUP BY invoice_year, invoice_month, store_id, manager_name
ORDER BY invoice_year DESC, invoice_month DESC, total_revenue DESC
LIMIT 20;